# Análisis de palabras con TF-IDF

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter

In [ ]:
file = "un-general-debates-blueprint.csv"

df = pd.read_csv("./dataset/"+file)
df.sample(2)

## Procesamiento

In [ ]:
import regex as re
import nltk

stopwords = set(nltk.corpus.stopwords.words('english'))

def tokenize(text):
    return re.findall(r'[\w-]*\p{L}[\w-]*',text)

def remove_stopword(tokens):
    return [t for t in tokens if t.lower() not in stopwords]

include_stopwords = {'dear', 'regards', 'must', 'would', 'also'}
exclude_stopwords = {'against'}

stopwords |= include_stopwords
stopwords -= exclude_stopwords

pipeline = [str.lower, tokenize, remove_stopword]

def prepare(text, pipeline):
    tokens = text
    for transform in pipeline:
        tokens = transform(tokens)
    return tokens

In [ ]:
df['tokens'] = df['text'].apply(prepare, pipeline= pipeline)
df['num_tokens'] = df['tokens'].map(len)

Para calcular TF-IDF del corpus es necesario volver a realizar el conteo de palabras

In [ ]:
def count_words(df, column='tokens', preprocess=None, min_freq=2):
    # process tokens and update counter
    def update(doc):
        tokens = doc if preprocess is None else preprocess(doc)
        counter.update(tokens)

    # create counter and run through all data
    counter = Counter()
    df[column].map(update)

    # transform counter into a DataFrame
    freq_df = pd.DataFrame.from_dict(counter, orient='index', columns=['freq'])
    freq_df = freq_df.query('freq >= @min_freq')
    freq_df.index.name = 'token'

    return freq_df.sort_values('freq', ascending=False)    

In [ ]:
freq_df = count_words(df)
freq_df.head(5)

# Cálculo de IDF

In [ ]:
import numpy as np

In [ ]:
def compute_idf(df, column='tokens', preprocess=None, min_freq=2):
    # process tokens and update counter
    def update(doc):
        tokens = doc if preprocess is None else preprocess(doc)
        counter.update(set(tokens))

    # create counter and run through all data
    counter = Counter()
    df[column].map(update)

    # transform counter into a DataFrame
    idf_df = pd.DataFrame.from_dict(counter, orient='index', columns=['df'])
    #idf_df = idf_df.query('df >= @min_freq')
    idf_df = idf_df.loc[idf_df['df'] >= min_freq]
    idf_df['idf'] = np.log(len(df)/idf_df['df']) + 0.1
    idf_df.index.name = 'token'

    return idf_df  

In [ ]:
idf_df = compute_idf(df)

In [ ]:
idf_df

A continuación calculamos TF-IDF



In [ ]:
freq_df['tfidf'] = freq_df['freq'] * idf_df['idf']

In [ ]:
freq_df.head()

## Representación visual de TF-IDF

In [ ]:
freq_1970 = count_words(df[df['year'] == 1970])
freq_1970['tfidf'] = freq_1970['freq'] * idf_df['idf']

In [ ]:
freq_1970

## Nube de palabras (wordcloud)

In [ ]:
from wordcloud import WordCloud

In [ ]:
def wordcloud(word_freq, title=None, max_words=200, stopwords=None):
    wc = WordCloud(width=800, height=400,
                   background_color='black', colormap='Paired',
                   max_font_size=150, max_words=max_words)
    
    # covert DataFrame into dict
    if type(word_freq) == pd.Series:
        counter = Counter(word_freq.fillna(0).to_dict())
    else:
        counter = word_freq

    # filter stop words in frequency counter
    if stopwords is not None:
        counter = {token:freq for (token, freq) in counter.items() if token not in stopwords}

    wc.generate_from_frequencies(counter)
    plt.title(title)
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')

In [ ]:
wordcloud(freq_1970['freq'], title='1970 - Frecuencia', stopwords=freq_1970.head(50))


In [ ]:
wordcloud(freq_1970['tfidf'], title='1970 - TFIDF', stopwords=["twenty-fifth", "twenty-five"])

## Ejercicio 2

A partir del UN General Debate Dataset, genera las nubes de palabras a partir de los resultados obtenidos en TF-IDF correspondientes a los años 1970, 1980, 1990, 2000 y 2010.


# N-gramas



In [ ]:
def ngrams(tokens, n=2, sep=' ', stopwords=set()):
    return [sep.join(ngram) for ngram in zip(*[tokens[i:] for i in range(n)]) if len([t for t in ngram if t in stopwords ]) == 0]

In [ ]:
df.head()

In [ ]:
df['bigrams'] = (
    df['text']
        .apply(
            prepare, 
            pipeline=[str.lower, tokenize]
        )
        .apply(
        ngrams, 
        n=2, 
        stopwords=stopwords
    )
)

df.head()

In [ ]:
count_words(df, 'bigrams')

In [ ]:
idf_df = (
    pd.concat(
        [idf_df, compute_idf(df, 'bigrams')]        
    )
)

In [ ]:
idf_df

In [ ]:
freq_df = count_words(df[df['year'] == 2015], 'bigrams')
freq_df['tfidf'] = freq_df['freq'] * idf_df['idf']

In [ ]:
wordcloud(freq_df['tfidf'], title='all bigrams', max_words=50)

## Ejercicio 3

A partir del UN General Debate Dataset, genera las nubes de palabras a partir de los resultados obtenidos con bigramas correspondientes a los años 1970, 1980, 1990, 2000 y 2010.

# Creación líneas de tiempo de frecuencia



In [ ]:
from collections import Counter

In [ ]:
def count_keywords(tokens, keywords):
    tokens = [t for t in tokens if t in keywords]
    counter = Counter(tokens)
    return [counter.get(k, 0) for k in keywords]

In [ ]:
keyword = ['nuclear', 'terrorism', 'climate', 'freedom']
token = ['nuclear', 'climate', 'climate', 'freedom', 'climate', 'freedom']

count_keywords(token, keyword)

In [ ]:
def count_keywords_by(df, by, keywords, column='tokens'):
    freq_matrix = df[column].apply(count_keywords, keywords=keywords)
    freq_df = pd.DataFrame.from_records(freq_matrix, columns=keywords)
    freq_df[by] = df[by]

    return freq_df.groupby(by=by).sum().sort_values(by)

In [ ]:
freq_df = count_keywords_by(df, by='year', keywords=keyword)
freq_df

In [ ]:
freq_df.plot(kind='line')

In [ ]:
import seaborn as sns

In [ ]:
freq_df = freq_df.div(df.groupby('year')['num_tokens'].sum(), axis=0)
freq_df.apply(np.sqrt)

sns.heatmap(data=freq_df, cmap='Reds')